# Executive Summary – Local Test Notebook

This notebook is **for local testing only**.  
It sets environment variables that `executive_summary_tab.py` reads when
Airflow is not available, then calls the exact same functions that the
production dashboard calls.

**Steps**
1. Cell 1 – install local dependencies (run once if needed)
2. Cell 2 – enter GP password and OpenAI key when prompted → env vars are set
3. Cell 3 – import the shared module (auto-detects local mode)
4. Cell 4 – choose a month
5. Cell 5 – query GP and inspect data
6. Cell 6 – generate the summary
7. Cell 7 – save to .txt (optional)

In [ ]:
# ── Cell 1: Install local dependencies (run once) ─────────────────────────
# Uncomment if packages are missing:
# !pip install psycopg2-binary openai pandas

In [ ]:
# ── Cell 2: Set credentials via environment variables ─────────────────────
#
# executive_summary_tab.py reads these when Airflow is not available.
# Set them BEFORE importing the module.

import os
import getpass

os.environ["GP_HOST"]     = "greenplum-rdsp.zur.swissbank.com"
os.environ["GP_PORT"]     = "5432"
os.environ["GP_DB"]       = "gprdsp"
os.environ["GP_USER"]     = "ds_rdsp_dev"
os.environ["GP_SCHEMA"]   = "core_ikg"
os.environ["GP_PASSWORD"] = getpass.getpass("Greenplum password: ")

os.environ["OPENAI_API_KEY"]  = getpass.getpass("OpenAI / Azure API key: ")
os.environ["OPENAI_BASE_URL"] = (
    "https://cirruspl-staat-ste-dev-ai.openai.azure.com/openai/v1/"
)

print("Credentials stored in environment variables.")

In [ ]:
# ── Cell 3: Import the shared module ──────────────────────────────────────
#
# Because Airflow is not installed locally, the module automatically enters
# local mode and reads credentials from the env vars set above.
#
# Place this notebook in the same directory as executive_summary_tab.py,
# or add its parent directory to sys.path.

import importlib
import sys

# Uncomment and adjust if the module is in a different directory:
# sys.path.insert(0, "/path/to/tabs")

import executive_summary_tab as est
importlib.reload(est)   # ensures fresh env vars are picked up if re-running

print(f"Airflow available : {est._AIRFLOW_AVAILABLE}")
print(f"Dash available    : {est._DASH_AVAILABLE}")
print("Module loaded in local mode – using GP_* env vars for connections.")

In [ ]:
# ── Cell 4: Choose a month ────────────────────────────────────────────────
#
# Lists available months from GP, then prompts for selection.

from datetime import datetime
import pandas as pd

month_options = est.get_month_options_from_db()

print("Available months (latest first):")
for i, opt in enumerate(month_options, start=1):
    print(f"  {i:>2}. {opt['label']}  ({opt['value']})")

print()
raw_input = input(
    "Enter the month to summarise (e.g. 'May 2026' or '2026-05'): "
).strip()

# Parse flexible input formats
try:
    SELECTED_MONTH = str(pd.Period(raw_input, "M"))
except Exception:
    cleaned = raw_input.replace("-", " ").replace("/", " ")
    dt = datetime.strptime(cleaned, "%B %Y")
    SELECTED_MONTH = dt.strftime("%Y-%m")

SELECTED_LABEL = pd.Period(SELECTED_MONTH, "M").to_timestamp().strftime("%B %Y")
print(f"\nSelected: {SELECTED_LABEL}  [{SELECTED_MONTH}]")

In [ ]:
# ── Cell 5: Query GP and inspect data ─────────────────────────────────────
#
# Calls the same get_exec_summary_data() that the Dash callback calls.

df_current, df_next_month = est.get_exec_summary_data(SELECTED_MONTH)

NEXT_LABEL = (
    pd.Period(SELECTED_MONTH, "M") + 1
).to_timestamp().strftime("%B %Y")

print(f"Current month ({SELECTED_LABEL}) rows : {len(df_current)}")
print(f"Next month    ({NEXT_LABEL}) rows : {len(df_next_month) if df_next_month is not None else 0}")
print()

display(df_current.head(10))

print("\nLabel breakdown:")
display(
    df_current["labels"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

print("\nRows labelled 'Top Feature':")
display(
    df_current[
        df_current["labels"].fillna("").apply(lambda v: est._has_label(v, "Top Feature"))
    ][["id_x", "title", "labels", "rule_name", "change_type"]]
)

print("\nRows labelled 'New Insight':")
display(
    df_current[
        df_current["labels"].fillna("").apply(lambda v: est._has_label(v, "New Insight"))
    ][["id_x", "title", "labels", "rule_name", "change_type"]]
)

In [ ]:
# ── Cell 6: Generate the executive summary ────────────────────────────────
#
# Calls the same generate_summary() that the Dash callback calls.

print(f"Calling {est.MODEL_NAME} … this may take up to 30 seconds.")

SUMMARY_TEXT = est.generate_summary(df_current, next_month_df=df_next_month)

print("\n" + "=" * 80)
print(f"Executive Summary – {SELECTED_LABEL}")
print("=" * 80 + "\n")
print(SUMMARY_TEXT)

In [ ]:
# ── Cell 7: Save to .txt file (optional) ──────────────────────────────────
#
# Calls the same build_save_content() that the Dash 'Save as TXT' button calls.

content  = est.build_save_content(SUMMARY_TEXT, SELECTED_LABEL)
filename = f"executive_summary_{SELECTED_MONTH}.txt"

with open(filename, "w", encoding="utf-8") as f:
    f.write(content)

print(f"Saved → {filename}")